# Banking Customer Support Intelligence (Google Colab)

End-to-end demo: data audit, EDA, intent classification, evaluation, confidence-based escalation, response drafts, conversation summaries, and an interactive notebook UI. Uses the supplied `dataset-20261112T050425Z-1-001.zip` (train/test CSVs) or separate CSV uploads.

**Safety:** use only public/synthetic examples. Never upload real customer messages, account data, card numbers, passwords, PINs, or OTPs. This prototype is not connected to any bank and must not be used for live customer decisions.


## Step 1 - Install dependencies
Run this cell once per new Colab runtime.


In [ ]:
%pip -q install pandas numpy scikit-learn matplotlib seaborn ipywidgets
import os, re, json, zipfile, shutil, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             top_k_accuracy_score, classification_report, ConfusionMatrixDisplay)
from IPython.display import display, Markdown

RANDOM_STATE = 42
WORK = Path('/content/banking_support_project')
WORK.mkdir(parents=True, exist_ok=True)
print('Workspace:', WORK)


## Step 2 - Upload and load the supplied dataset
Upload the supplied ZIP, or upload both `train.csv` and `test.csv`. The loader searches for `dataset/train.csv` and `dataset/test.csv` inside the archive.


In [ ]:
from google.colab import files
uploaded = files.upload()  # choose the dataset ZIP, or both CSV split files

zip_names = [n for n in uploaded if n.lower().endswith('.zip')]
csv_names = [n for n in uploaded if n.lower().endswith('.csv')]
if zip_names:
    with zipfile.ZipFile(zip_names[0]) as zf:
        members = zf.namelist()
        train_member = next((n for n in members if n.lower().endswith('train.csv')), None)
        test_member = next((n for n in members if n.lower().endswith('test.csv')), None)
        if not train_member or not test_member:
            raise FileNotFoundError('ZIP must contain train.csv and test.csv')
        (WORK / 'train.csv').write_bytes(zf.read(train_member))
        (WORK / 'test.csv').write_bytes(zf.read(test_member))
elif len(csv_names) >= 2:
    train_name = next((n for n in csv_names if 'train' in n.lower()), csv_names[0])
    test_name = next((n for n in csv_names if 'test' in n.lower()), next(n for n in csv_names if n != train_name))
    shutil.copy(train_name, WORK / 'train.csv')
    shutil.copy(test_name, WORK / 'test.csv')
else:
    raise FileNotFoundError('Upload the supplied ZIP or both train.csv and test.csv')

REQUIRED = {'text', 'category'}
def load_split(path):
    frame = pd.read_csv(path)
    missing = REQUIRED - set(frame.columns)
    if missing:
        raise ValueError(f'{path.name} is missing columns: {sorted(missing)}')
    frame = frame[['text', 'category']].copy()
    frame['text'] = frame['text'].fillna('').astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()
    frame['category'] = frame['category'].fillna('').astype(str).str.strip()
    return frame.loc[frame.text.ne('') & frame.category.ne('')].reset_index(drop=True)

train_df = load_split(WORK / 'train.csv')
test_df = load_split(WORK / 'test.csv')
assert set(test_df.category).issubset(set(train_df.category)), 'Test set contains a label not present in training.'
print('Train shape:', train_df.shape, '| Test shape:', test_df.shape)
print('Train intents:', train_df.category.nunique(), '| Test intents:', test_df.category.nunique())
display(train_df.head())


## Step 3 - Data audit and exploratory analysis
These checks report what is in the uploaded files. The source split remains separate; a validation slice is made from training only.


In [ ]:
audit = {
    'train_rows_after_cleanup': len(train_df),
    'test_rows_after_cleanup': len(test_df),
    'train_missing_text': int(train_df.text.isna().sum()),
    'train_blank_text': int(train_df.text.str.strip().eq('').sum()),
    'train_duplicate_text_rows': int(train_df.text.duplicated().sum()),
    'train_test_exact_text_overlap': int(len(set(train_df.text) & set(test_df.text))),
    'train_intents': int(train_df.category.nunique()),
    'test_intents': int(test_df.category.nunique()),
    'train_class_counts': train_df.category.value_counts().sort_index().to_dict(),
    'test_class_counts': test_df.category.value_counts().sort_index().to_dict(),
}
print(json.dumps(audit, indent=2))
(WORK / 'data_audit.json').write_text(json.dumps(audit, indent=2), encoding='utf-8')

counts = train_df.category.value_counts().sort_values()
plt.figure(figsize=(10, max(8, len(counts) * .18)))
counts.plot(kind='barh', color='#2457A7')
plt.title('Training examples by intent'); plt.xlabel('Examples'); plt.ylabel('Intent'); plt.tight_layout(); plt.show()

train_df['word_count'] = train_df.text.str.split().str.len()
plt.figure(figsize=(8, 4)); sns.histplot(train_df.word_count, bins=30, color='#17A398')
plt.title('Customer query length'); plt.xlabel('Words'); plt.tight_layout(); plt.show()
display(train_df.groupby('category').word_count.agg(['count','mean','median']).sort_values('count').head(15))


## Step 4 - Train and select machine-learning baselines
Word and character TF-IDF features capture both phrase meaning and spelling variation. Compare Logistic Regression and calibrated Linear SVM on a stratified validation slice; the supplied test split is kept out of model selection.


In [ ]:
def make_features():
    return FeatureUnion([
        ('word', TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True,
                                strip_accents='unicode', max_features=250_000)),
        ('char', TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=1,
                                sublinear_tf=True, max_features=300_000)),
    ])

def make_pipeline(kind):
    estimator = (LogisticRegression(max_iter=1200, C=4, solver='lbfgs') if kind == 'logistic'
                 else LinearSVC(C=1.5))
    return Pipeline([('features', make_features()), ('classifier', estimator)])

def probabilities(model, texts):
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(texts)
    scores = model.decision_function(texts)
    scores = scores - scores.max(axis=1, keepdims=True)
    exp_scores = np.exp(np.clip(scores, -50, 50))
    return exp_scores / exp_scores.sum(axis=1, keepdims=True)

X_train, X_val, y_train, y_val = train_test_split(
    train_df.text, train_df.category, test_size=.15, random_state=RANDOM_STATE, stratify=train_df.category)
labels = np.array(sorted(train_df.category.unique()))
validation_results, candidates = [], {}
for name in ['logistic', 'linear_svc']:
    base = make_pipeline(name)
    if name == 'linear_svc':
        candidate = CalibratedClassifierCV(base, method='sigmoid', cv=3).fit(X_train, y_train)
    else:
        candidate = base.fit(X_train, y_train)
    pred = candidate.predict(X_val)
    probs = probabilities(candidate, X_val)
    row = {'model': name, 'accuracy': accuracy_score(y_val, pred),
           'macro_f1': f1_score(y_val, pred, average='macro', zero_division=0),
           'weighted_f1': f1_score(y_val, pred, average='weighted', zero_division=0),
           'macro_precision': precision_score(y_val, pred, average='macro', zero_division=0),
           'macro_recall': recall_score(y_val, pred, average='macro', zero_division=0),
           'top3_accuracy': top_k_accuracy_score(y_val, probs, k=3, labels=labels)}
    validation_results.append(row); candidates[name] = candidate
validation_df = pd.DataFrame(validation_results).sort_values('macro_f1', ascending=False)
display(validation_df)
validation_df.to_csv(WORK / 'validation_metrics.csv', index=False)
best_name = validation_df.iloc[0]['model']
print('Selected by validation macro-F1:', best_name)

# Refit the selected pipeline on all of train.csv. test.csv is still untouched.
if best_name == 'linear_svc':
    model = CalibratedClassifierCV(make_pipeline('linear_svc'), method='sigmoid', cv=3).fit(train_df.text, train_df.category)
else:
    model = make_pipeline('logistic').fit(train_df.text, train_df.category)

import joblib
joblib.dump(model, WORK / 'intent_model.joblib')
print('Saved:', WORK / 'intent_model.joblib')


## Step 5 - Final evaluation on held-out test data
Run only after model selection. These are measured results from this dataset/runtime; do not copy a score into the report until this cell finishes.


In [ ]:
y_test = test_df.category
test_probs = probabilities(model, test_df.text)
test_pred = model.predict(test_df.text)
test_metrics = {
    'selected_model': best_name,
    'accuracy': float(accuracy_score(y_test, test_pred)),
    'macro_f1': float(f1_score(y_test, test_pred, average='macro', zero_division=0)),
    'weighted_f1': float(f1_score(y_test, test_pred, average='weighted', zero_division=0)),
    'macro_precision': float(precision_score(y_test, test_pred, average='macro', zero_division=0)),
    'macro_recall': float(recall_score(y_test, test_pred, average='macro', zero_division=0)),
    'top3_accuracy': float(top_k_accuracy_score(y_test, test_probs, k=3, labels=labels)),
}
print(json.dumps(test_metrics, indent=2))
(WORK / 'test_metrics.json').write_text(json.dumps(test_metrics, indent=2), encoding='utf-8')
report = pd.DataFrame(classification_report(y_test, test_pred, output_dict=True, zero_division=0)).T
display(report.sort_values('f1-score').head(20))
report.to_csv(WORK / 'classification_report.csv')
predictions = pd.DataFrame({'text': test_df.text, 'actual': y_test, 'predicted': test_pred, 'confidence': test_probs.max(axis=1)})
predictions.to_csv(WORK / 'test_predictions.csv', index=False)

# Show a compact confusion matrix for the 10 most frequent test intents.
top_labels = y_test.value_counts().head(10).index.tolist()
subset = test_df[y_test.isin(top_labels)].copy()
fig, ax = plt.subplots(figsize=(11, 9))
ConfusionMatrixDisplay.from_predictions(subset.category, model.predict(subset.text), labels=top_labels,
    xticks_rotation=90, cmap='Blues', colorbar=False, ax=ax)
ax.set_title('Confusion matrix - 10 most frequent test intents'); plt.tight_layout(); plt.show()


## Step 6 - Confidence routing and safe response drafts
Thresholds here are demonstration heuristics, not calibrated safety guarantees. Fraud cues and low or medium confidence go to a human. Drafts never claim to have accessed or changed an account.


In [ ]:
HIGH_THRESHOLD, MEDIUM_THRESHOLD = .80, .55
URGENT_CUES = ('fraud', 'scam', 'stolen', 'unauthori', 'not recognised', 'not recognized', 'not_recogn')
GENERIC_RESPONSE = (
    "I can help route this question to the right support team. Please use your bank's official app or verified website for account-specific help. "
    'Do not share passwords, PINs, one-time passcodes, or full card numbers in chat.'
)

def assess_query(text, category, confidence):
    context = f'{text} {category}'.lower()
    if any(cue in context for cue in URGENT_CUES):
        return {'band':'urgent', 'escalate':True, 'reason':'Possible fraud or unauthorized activity; route to trained human support immediately.'}
    if confidence >= HIGH_THRESHOLD:
        return {'band':'high', 'escalate':False, 'reason':'High model confidence; agent review is still required.'}
    if confidence >= MEDIUM_THRESHOLD:
        return {'band':'medium', 'escalate':True, 'reason':'Agent should verify intent before using any draft.'}
    return {'band':'low', 'escalate':True, 'reason':'Low confidence; do not rely on the prediction.'}

def response_draft(category, confidence, urgent=False):
    if urgent or confidence < MEDIUM_THRESHOLD:
        return GENERIC_RESPONSE
    key = category.lower()
    if any(x in key for x in ('fraud','scam','not_recogn','not recognised','not recognized')):
        return 'If you suspect an unauthorized transaction, lock the affected card in the official bank app if available and contact the bank immediately through a verified channel. Never share credentials or one-time passcodes.'
    if 'cash_withdrawal' in key or 'cash machine' in key or 'atm' in key:
        return 'Check the transaction in your official banking app. If cash was not dispensed or the amount looks wrong, contact verified bank support with the date, amount, and ATM location. Never share your PIN or OTP.'
    if 'card' in key:
        return "Check card status and controls in your bank's official app. For a missing or unrecognized card, contact verified bank support. Never share your PIN, OTP, password, or full card number."
    return GENERIC_RESPONSE

def classify_query(text):
    p = probabilities(model, [text])[0]
    order = np.argsort(p)[::-1]
    category, score = model.classes_[order[0]], float(p[order[0]])
    routing = assess_query(text, category, score)
    draft = response_draft(category, score, urgent=(routing['band'] == 'urgent'))
    alternatives = pd.DataFrame({'intent':[model.classes_[i] for i in order[:5]], 'score':[float(p[i]) for i in order[:5]]})
    return category, score, routing, draft, alternatives

demo_query = 'My card has not arrived after two weeks.'
category, score, routing, draft, alternatives = classify_query(demo_query)
print('Intent:', category, '| Confidence:', f'{score:.1%}', '| Routing:', routing)
print('Response draft:', draft)
display(alternatives)


## Step 7 - Conversation summary
The provided dataset is intent utterances, not multi-turn conversations. This simple extractive demonstration selects distinct sentences; it is not a validated summarization model.


In [ ]:
def summarize_conversation(messages, max_sentences=3):
    sentences = [s.strip() for msg in messages for s in re.split(r'(?<=[.!?])\s+', str(msg)) if s.strip()]
    selected, seen = [], set()
    for sentence in sentences:
        normalized = re.sub(r'\W+', ' ', sentence.lower()).strip()
        if len(normalized) < 12 or normalized in seen:
            continue
        seen.add(normalized); selected.append(sentence)
        if len(selected) >= max(1, max_sentences):
            break
    return ' '.join(selected) if selected else 'No conversation text was provided.'

sample_messages = [
    'Customer: My card has not arrived. I ordered it two weeks ago.',
    'Agent: I can help check the delivery steps. Please do not share your card number or OTP here.'
]
print(summarize_conversation(sample_messages))


## Step 8 - Interactive query demo
Try only synthetic/public examples. The widget does not send or store messages outside this Colab runtime.


In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output
query_box = widgets.Textarea(value='', placeholder='Enter a synthetic, non-sensitive example', description='Query:', layout=widgets.Layout(width='90%', height='100px'))
run_button = widgets.Button(description='Classify and draft', button_style='primary')
output = widgets.Output()
def on_run(_):
    with output:
        clear_output()
        if not query_box.value.strip():
            print('Enter a sample query first.'); return
        cat, conf, route, answer, choices = classify_query(query_box.value.strip())
        print(f'Intent: {cat} | Confidence: {conf:.1%} | Route: {route["band"]}')
        print('Escalate:', route['escalate'], '|', route['reason'])
        print('Draft:', answer)
        display(choices)
run_button.on_click(on_run)
display(query_box, run_button, output)


## Step 9 - Optional deep-learning comparison (BiLSTM)
Run this only if you want the brief's deep-learning comparison. It may take several minutes. The baseline remains the main model.


In [ ]:
# Optional: uncomment and run after the baseline cells.
# import tensorflow as tf
# from sklearn.preprocessing import LabelEncoder
# tf.keras.utils.set_random_seed(RANDOM_STATE)
# encoder = LabelEncoder().fit(train_df.category)
# max_tokens, sequence_len = 30000, 60
# vectorizer = tf.keras.layers.TextVectorization(max_tokens=max_tokens, output_mode='int', output_sequence_length=sequence_len, standardize='lower_and_strip_punctuation')
# vectorizer.adapt(train_df.text.to_numpy())
# inputs = tf.keras.Input(shape=(), dtype=tf.string)
# x = vectorizer(inputs)
# x = tf.keras.layers.Embedding(max_tokens, 96, mask_zero=True)(x)
# x = tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, dropout=.2))(x)
# x = tf.keras.layers.Dense(128, activation='relu')(x)
# x = tf.keras.layers.Dropout(.3)(x)
# outputs = tf.keras.layers.Dense(len(encoder.classes_), activation='softmax')(x)
# bilstm = tf.keras.Model(inputs, outputs)
# bilstm.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
# Xa, Xv, ya, yv = train_test_split(train_df.text.to_numpy(), encoder.transform(train_df.category), test_size=.15, random_state=RANDOM_STATE, stratify=train_df.category)
# bilstm.fit(Xa, ya, validation_data=(Xv, yv), epochs=12, batch_size=64, callbacks=[tf.keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True)], verbose=2)
# dl_probs = bilstm.predict(test_df.text.to_numpy(), verbose=0)
# dl_pred = encoder.inverse_transform(dl_probs.argmax(axis=1))
# print(classification_report(test_df.category, dl_pred, zero_division=0))
# bilstm.save(WORK / 'intent_bilstm.keras')


## Step 10 - Create the Streamlit application file
This cell writes a Streamlit app beside the trained model. The final ZIP includes both so you can download them and run `streamlit run app.py` locally.


In [ ]:
app_source = r'''
import re
from pathlib import Path
import joblib
import pandas as pd
import streamlit as st

ROOT = Path(__file__).resolve().parent
MODEL_PATH = ROOT / 'intent_model.joblib'
st.set_page_config(page_title='Banking Support Intelligence', page_icon='🏦', layout='wide')
st.title('🏦 Banking Customer Support Intelligence')
st.caption('Intent prediction, human escalation, response drafts, and conversation summaries')
if not MODEL_PATH.exists():
    st.error('intent_model.joblib is missing. Extract the full Colab output ZIP beside app.py.')
    st.stop()
model = joblib.load(MODEL_PATH)

def route(text, category, score):
    context = f'{text} {category}'.lower()
    urgent = any(x in context for x in ('fraud','scam','stolen','unauthori','not recognised','not recognized','not_recogn'))
    if urgent: return True, 'Potential fraud or unauthorized activity: contact verified bank support immediately.'
    if score < .55: return True, 'Low confidence: human agent review required.'
    if score < .80: return True, 'Medium confidence: agent should verify the intent.'
    return False, 'High confidence; an agent must still review the draft before sending.'

def draft(category, score, urgent):
    if urgent or score < .55:
        return "Please use your bank's official app or verified website for account-specific help. Do not share passwords, PINs, one-time passcodes, or full card numbers in chat."
    key = category.lower()
    if any(x in key for x in ('fraud','scam','not_recogn')):
        return 'If you suspect unauthorized activity, lock the affected card in the official bank app if available and contact the bank immediately through a verified channel. Never share credentials or one-time passcodes.'
    if 'cash_withdrawal' in key or 'atm' in key:
        return 'Check the transaction in your official banking app. If cash was not dispensed or the amount looks wrong, contact verified support with the date, amount, and ATM location. Never share your PIN or OTP.'
    if 'card' in key:
        return "Check your card status in the bank's official app. Contact verified bank support about a missing or unrecognized card. Never share your PIN, OTP, password, or full card number."
    return "I can help route this question to the right support team. Please use your bank's official app or verified website. Never share passwords, PINs, one-time passcodes, or full card numbers in chat."

def summarize(text):
    parts = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]
    result, seen = [], set()
    for sentence in parts:
        key = re.sub(r'\W+', ' ', sentence.lower()).strip()
        if len(key) >= 12 and key not in seen:
            seen.add(key); result.append(sentence)
        if len(result) == 3: break
    return ' '.join(result) if result else 'No conversation text was provided.'

tab1, tab2, tab3 = st.tabs(['Assist a query', 'Summarize a conversation', 'Test analytics'])
with tab1:
    st.info('Use synthetic examples only. No messages are sent and no bank account is accessed.')
    query = st.text_area('Customer message', placeholder='Type a non-sensitive example')
    if st.button('Classify and draft', disabled=not query.strip()):
        probs = model.predict_proba([query])[0]; order = probs.argsort()[::-1]
        category = str(model.classes_[order[0]]); score = float(probs[order[0]])
        escalate, reason = route(query, category, score)
        c1,c2,c3 = st.columns(3)
        c1.metric('Predicted intent', category.replace('_',' ').title())
        c2.metric('Confidence', f'{score:.1%}')
        c3.metric('Human review', 'Required' if escalate else 'Before sending')
        st.warning(reason) if escalate else st.info(reason)
        st.markdown('**Suggested draft**'); st.write(draft(category, score, 'fraud' in query.lower() or 'scam' in query.lower()))
        display = pd.DataFrame({'Intent':[model.classes_[i] for i in order[:5]],'Score':[float(probs[i]) for i in order[:5]]})
        st.dataframe(display, hide_index=True, use_container_width=True)
with tab2:
    conversation = st.text_area('Paste a synthetic conversation', key='conversation')
    if st.button('Summarize', disabled=not conversation.strip()): st.write(summarize(conversation))
    st.caption('Extractive demo summary. Check it against the original before use.')
with tab3:
    metrics_path = ROOT / 'test_metrics.json'
    if metrics_path.exists():
        import json
        metrics = json.loads(metrics_path.read_text())
        cols = st.columns(5)
        for col, key in zip(cols, ('accuracy','macro_f1','weighted_f1','macro_precision','top3_accuracy')):
            col.metric(key.replace('_',' ').title(), f"{metrics[key]:.1%}")
    else: st.info('Run the Colab evaluation cell and download its output ZIP to see metrics here.')
st.caption('Educational prototype only. No live LLM, account access, transaction execution, or customer messaging.')
'''
(WORK / 'app.py').write_text(app_source, encoding='utf-8')
print('Created Streamlit app:', WORK / 'app.py')


## Step 11 - Save and download project outputs
The output ZIP includes the fitted model, generated Streamlit app, audit, validation metrics, test metrics, per-intent report, predictions, and uploaded dataset splits.
After downloading and extracting it locally, install `streamlit`, `pandas`, `scikit-learn`, and `joblib`, then run `streamlit run app.py`.


In [ ]:
print('Generated files:')
for path in sorted(WORK.iterdir()):
    print(path.name, f'({path.stat().st_size:,} bytes)')

# Download a ZIP of results from Colab to your computer.
from google.colab import files
output_zip = Path('/content/banking_support_project_outputs.zip')
with zipfile.ZipFile(output_zip, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for path in WORK.iterdir():
        if path.is_file(): zf.write(path, arcname=path.name)
files.download(str(output_zip))
